# Student 4 — Video Analyst: CCTV Surveillance Footage Processing

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 4 — Video Analyst  
**Pipeline:** Load CAVIAR clips → OpenCV frame extraction → Motion detection → YOLOv8 object detection → Event classification → Export CSV

### Required Output Columns
`Clip_ID | Timestamp | Frame_ID | Event_Detected | Persons_Count | Confidence`

---

## Dataset
**CAVIAR CCTV Dataset** — simulated indoor surveillance footage (people walking, fighting, collapsing)  
- Direct link: `http://homepages.inf.ed.ac.uk/rbf/CAVIARDATA1/`  
- No account or signup needed — download `.mpg` files directly  
- Recommended clips: `Browse`, `OneStopEnter` (basic motion), `Fight`, `Rest_FallOnFloor` (anomaly)

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
!pip install ultralytics opencv-python moviepy imageio pandas -q
print('All packages installed.')

In [ ]:
# ============================================================
# CELL 2 — Download CAVIAR clips (no login required)
# Tip: download only 3-5 short clips to start
# ============================================================
import urllib.request, os

BASE_URL = 'http://homepages.inf.ed.ac.uk/rbf/CAVIARDATA1'

# 3 clips: fight/anomaly scenarios + basic motion
CLIPS = [
    ('Fight_Chase',    'Fight_Chase.mpg'),
    ('Fight_RunAway1', 'Fight_RunAway1.mpg'),
    ('Browse1',        'Browse1.mpg'),
]

for folder, filename in CLIPS:
    if os.path.exists(filename):
        print(f'Already downloaded: {filename}')
        continue
    url = f'{BASE_URL}/{folder}/{filename}'
    print(f'Downloading {filename} from {url}...')
    urllib.request.urlretrieve(url, filename)
    size_mb = os.path.getsize(filename) / 1024 / 1024
    print(f'  Saved {filename} ({size_mb:.1f} MB)')

video_files = [f for f in os.listdir('.') if f.endswith('.mpg')]
print(f'\nReady: {video_files}')

In [ ]:
# ============================================================
# CELL 3 — Frame extraction with OpenCV
# Extract one frame every FRAME_INTERVAL frames
# ============================================================
import cv2
import os

FRAME_INTERVAL = 15   # sample every 15th frame (~0.6s at 25fps)
FRAMES_DIR     = 'frames'
os.makedirs(FRAMES_DIR, exist_ok=True)

clip_meta = {}  # {clip_file: {fps, total_frames, frame_paths}}

for clip_file in video_files:
    cap         = cv2.VideoCapture(clip_file)
    fps         = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    clip_name   = os.path.splitext(clip_file)[0]
    frame_paths = []
    frame_idx   = 0

    print(f'{clip_file}: {total} frames @ {fps:.1f} fps')

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_INTERVAL == 0:
            out_path = os.path.join(FRAMES_DIR, f'{clip_name}_FRM_{frame_idx:04d}.jpg')
            cv2.imwrite(out_path, frame)
            frame_paths.append((frame_idx, out_path))
        frame_idx += 1

    cap.release()
    clip_meta[clip_file] = {'fps': fps, 'total': total, 'frames': frame_paths}
    print(f'  Extracted {len(frame_paths)} frames')

In [ ]:
# ============================================================
# CELL 4 — Motion detection (OpenCV frame differencing)
# Computes motion score per sampled frame
# ============================================================
import cv2
import numpy as np

def compute_motion_score(prev_gray, curr_gray, threshold=25):
    """
    Returns motion_score in [0,1]: fraction of pixels that changed
    above the pixel-difference threshold.
    """
    if prev_gray is None:
        return 0.0
    diff   = cv2.absdiff(prev_gray, curr_gray)
    _, thr = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
    return float(thr.sum()) / (thr.size * 255)

motion_scores = {}  # {(clip_file, frame_idx): score}

for clip_file, meta in clip_meta.items():
    prev_gray = None
    for frame_idx, frame_path in meta['frames']:
        img  = cv2.imread(frame_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)
        score = compute_motion_score(prev_gray, gray)
        motion_scores[(clip_file, frame_idx)] = score
        prev_gray = gray

print(f'Motion scores computed for {len(motion_scores)} frames')
# Show top 5 highest motion frames
top5 = sorted(motion_scores.items(), key=lambda x: -x[1])[:5]
for (clip, fidx), score in top5:
    print(f'  {clip} frame {fidx}: motion={score:.4f}')

In [ ]:
# ============================================================
# CELL 5 — YOLOv8 object detection on extracted frames
# ============================================================
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # ~6MB nano model, runs on CPU

# Event classification using clip name + detected objects + motion
EVENT_MAP = [
    (['fire', 'smoke'],                              'Fire detected'),
    (['car', 'truck', 'motorcycle', 'bicycle'],      'Vehicle movement'),
    (['person'],                                     'Person detected'),
]

def classify_event(objects, motion_score, clip_name):
    cl = clip_name.lower()
    if ('fight' in cl or 'runaway' in cl or 'run' in cl) and motion_score > 0.15:
        return 'Fighting / Running'
    if 'fight' in cl or 'runaway' in cl or 'chase' in cl:
        return 'Suspicious movement'
    if 'collapse' in cl or 'fall' in cl or 'slump' in cl:
        return 'Person collapsing'
    if 'browse' in cl or 'shop' in cl:
        return 'Person browsing' if objects else 'Empty scene'
    obj_lower = [o.lower() for o in objects]
    for keywords, label in EVENT_MAP:
        if any(k in obj_lower for k in keywords):
            return label
    return 'No event'

yolo_results = {}  # {(clip_file, frame_idx): {detected, confs, persons}}

for clip_file, meta in clip_meta.items():
    clip_name = os.path.splitext(clip_file)[0]
    for frame_idx, frame_path in meta['frames']:
        results  = model(frame_path, verbose=False)
        r        = results[0]
        detected = [model.names[int(b.cls[0])] for b in r.boxes]
        confs    = [round(float(b.conf[0]), 2) for b in r.boxes]
        yolo_results[(clip_file, frame_idx)] = {
            'detected': detected,
            'confs':    confs,
            'persons':  detected.count('person'),
        }

print(f'YOLOv8 inference complete: {len(yolo_results)} frames processed')

In [ ]:
# ============================================================
# CELL 6 — Build structured DataFrame
# ============================================================
import pandas as pd

CLIP_ID_MAP = {
    'Fight_Chase.mpg':    'CAVIAR_01',
    'Fight_RunAway1.mpg': 'CAVIAR_02',
    'Browse1.mpg':        'CAVIAR_03',
}

rows = []
for clip_file, meta in clip_meta.items():
    clip_name = os.path.splitext(clip_file)[0]
    clip_id   = CLIP_ID_MAP.get(clip_file, clip_name)
    fps       = meta['fps']

    for frame_idx, frame_path in meta['frames']:
        det    = yolo_results.get((clip_file, frame_idx), {})
        motion = motion_scores.get((clip_file, frame_idx), 0.0)

        detected = det.get('detected', [])
        confs    = det.get('confs', [])
        persons  = det.get('persons', 0)
        avg_conf = round(sum(confs) / max(len(confs), 1), 2)
        event    = classify_event(detected, motion, clip_name)

        seconds   = frame_idx / fps
        timestamp = f'{int(seconds//60):02d}:{int(seconds%60):02d}'

        rows.append({
            'Clip_ID':        clip_id,
            'Timestamp':      timestamp,
            'Frame_ID':       f'FRM_{frame_idx:04d}',
            'Event_Detected': event,
            'Persons_Count':  persons,
            'Confidence':     avg_conf,
        })

video_df = pd.DataFrame(rows)
print(f'Total rows: {len(video_df)}')
print(video_df['Event_Detected'].value_counts())
print()
print(video_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# CELL 7 — Validate and save video_output.csv
# ============================================================

REQUIRED_COLUMNS = ['Clip_ID', 'Timestamp', 'Frame_ID', 'Event_Detected', 'Persons_Count', 'Confidence']

video_df = video_df[REQUIRED_COLUMNS]

assert list(video_df.columns) == REQUIRED_COLUMNS, f'Column mismatch: {list(video_df.columns)}'
assert len(video_df) > 0, 'DataFrame is empty!'

video_df.to_csv('video_output.csv', index=False)

print(f'Saved video_output.csv')
print(f'  Rows: {len(video_df)} | Columns: {REQUIRED_COLUMNS}')
print()
print(video_df.to_string(index=False))